# Rețele neuronale și deep learning — Seminarul 7: Atenția și transformerele
Prof. dr. Daniel Traian Pele — Academia de Studii Economice din București (ASE); Academia Română, Institutul de Prognoză Economică (IPE)

[Deschide în Colab](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/notebooks/L7_seminar_ro.ipynb)

Sarcinile seminarului, pe datele reale din `../data/clean` (vezi `../data/SOURCES.md`). Scrieți codul în celulele goale de sub fiecare sarcină; păstrați valoarea fixă a generatorului aleator, ca rezultatele să poată fi comparate.

In [ ]:
# Google Colab: copiem repo-ul cursului, ca datele și rezultatele salvate să fie găsite
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/NEURAL_BIZ'):
    if not os.path.isdir('/content/NEURAL_BIZ'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/NEURAL_BIZ.git', '/content/NEURAL_BIZ'], check=True)
    os.chdir('/content/NEURAL_BIZ/notebooks')

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DATA = '../data/clean/'
SEED = 20261108
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
except ImportError:
    DEVICE = 'cpu'
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'axes.grid': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE)

## Pregătire

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tw = pd.read_csv(DATA + 'twitter_fin_sentiment.csv')            # 0 bearish, 1 bullish, 2 neutral
tr_all, te = tw[tw.split == 'train'].reset_index(drop=True), tw[tw.split == 'valid'].reset_index(drop=True)
tr, va = train_test_split(tr_all, test_size=0.10, stratify=tr_all.label, random_state=SEED)
tr, va = tr.reset_index(drop=True), va.reset_index(drop=True)
ytr, yva, yte = tr.label.values, va.label.values, te.label.values
tok = AutoTokenizer.from_pretrained('ProsusAI/finbert')
def enc(texts, n=64):
    e = tok(list(texts), padding='max_length', truncation=True, max_length=n, return_tensors='pt')
    return e['input_ids'], e['attention_mask']
Itr, Mtr = enc(tr.text); Iva, Mva = enc(va.text); Ite, Mte = enc(te.text)
print('train', len(tr), 'validation', len(va), 'test', len(te), ' vocabulary', tok.vocab_size)

**Sarcina 1.** Modelul de referință TF-IDF: comparați unigramele cu unigramele + bigramele (regresie logistică, $C=10$, ponderi de clasă echilibrate). Raportați macro-F1 și recall-ul clasei negative pe tweet-urile de test.

In [ ]:
# codul vostru aici

**Sarcina 2.** Atenția calculată manual: pentru primul tweet de test, luați intrarea primului strat FinBERT și proiecțiile query și key ale capului 3; calculați matricea de atenție și verificați că este egală cu cea dată de model.

In [ ]:
# codul vostru aici

**Sarcina 3.** De ce $\sqrt{d_k}$? Cu aceleași proiecții, calculați abaterea standard a scorurilor brute $\mathbf q^\top\mathbf k$ și a celor scalate, precum și cea mai mare pondere de atenție din fiecare rînd, cu și fără scalare.

In [ ]:
# codul vostru aici

**Sarcina 4.** FinBERT fără antrenare: asociați etichetele lui (positive, negative, neutral) cu (pozitiv, negativ, neutru) și calculați matricea de confuzie pe tweet-urile de test. Care eroare este cea mai frecventă?

In [ ]:
# codul vostru aici

**Sarcina 5.** Documente lungi: împărțiți cele trei cele mai lungi discursuri ale Băncii Centrale Europene în bucăți de 510 tokeni, dați un scor fiecărei bucăți cu FinBERT fără antrenare și raportați ponderea bucăților negative în fiecare discurs.

In [ ]:
# codul vostru aici